In [1]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import utils2

In [2]:
sub_list = sorted([int(os.path.basename(path).split('-')[1]) for path in 
            glob(os.path.join(utils2.DATASET_PATH, 'sub*'))])

print(sub_list)
print(len(sub_list))

[20131, 20209, 20725, 21100, 21214, 21508, 22172, 22268, 22281, 22286, 22295, 22346, 22353, 22354, 22365, 22366, 22380, 22381, 22475, 22477, 22482, 22487, 22494, 22532, 22536, 22587, 22600, 22623, 22630, 22686, 22687, 22727, 22734, 22818, 22829, 22831, 22847, 22860, 22862, 22871, 22873, 22906, 22956, 23010, 23014, 23028, 23044, 23072, 23123, 23257, 23261, 23294, 23303, 23314, 23317, 23342, 23353, 23362, 23376, 23381, 23382, 23386, 23392, 23407, 23432, 23442, 23480, 23483, 23485, 23494, 23501, 23524, 23532, 23534, 23545, 23559, 23567, 23568, 23572, 23584, 23601, 23622, 23629, 23631, 23664, 23690, 23735, 23749, 23762, 23779, 23822, 23842, 23855, 23865, 23871, 23878, 23883, 23892, 23910, 23923, 23925, 23946, 23971, 23972, 23979, 23985, 23986, 23987, 23991, 23996, 24004, 24015, 24017, 24021, 24022, 24029, 24033, 24038, 24039, 24040, 24046, 24050, 24067, 24073, 24089, 24091, 24106, 24115, 24118, 24127, 24139, 24142, 24145, 24146, 24148, 24151, 24182, 24197, 24198, 24212, 24217, 24236, 24240

In [3]:
PHENOTYPE_PATH = os.path.join(utils2.DATASET_PATH, 'phenotype', 'phenotype_preprocessed.tsv')
phenotype = pd.read_csv(PHENOTYPE_PATH, delimiter='\t')
phenotype.head()

,participant_id,sex,age_baseline,KSADS_MAIN_DIAGNOSIS,WASI_FULL_2_IQ,INCOME,RACE_WHITE,RACE_BLACK,RACE_ASIAN,RACE_MULTIPLE,ETHNICITY,COHORT,SCANNER,SCARED_BASELINE_YOUTH_RATING,SCARED_BASELINE_PARENT_RATING,CGAS_BASELINE_SCORE,PARS_BASELINE_CLINICIAN_RATING,IQ_IS_MISSING,INCOME_IS_MISSING
0,sub-020131,1.0,17.0,HV,122.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,4.0,7.2,NaN,NaN,-1,-1
1,sub-020209,1.0,16.0,HV,118.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,23.0,17.0,NaN,NaN,-1,-1
2,sub-020725,1.0,14.0,HV,116.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,3.0,0.0,NaN,NaN,-1,-1
3,sub-021100,-1.0,11.0,HV,134.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,8.0,1.0,NaN,NaN,-1,-1
4,sub-021214,1.0,11.0,HV,130.0,9.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,16.0,1.0,NaN,NaN,-1,-1


In [4]:
subs_info = utils2.get_sub_info_list(sub_list, phenotype)
subs_info

[{'subject_id': '20131', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '20209', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '20725', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '21100', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '21214', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '21508', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22172', 'cohort': 2, 'diagnosis': 'Healthy'},
 {'subject_id': '22268', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22281', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22286', 'cohort': 1, 'diagnosis': 'Anxiety'},
 {'subject_id': '22295', 'cohort': 1, 'diagnosis': 'Anxiety'},
 {'subject_id': '22346', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22353', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22354', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22365', 'cohort': 1, 'diagnosis': 'Healthy'},
 {'subject_id': '22366', 'cohort': 1, 'diagnosis': 'Hea

In [5]:
for sub_info in subs_info:
    print(f'sub-{sub_info['subject_id']}')
    # get fmri maps & tables
    runs, mean_fmri, rest, _, runs_confounds_tables, rest_confounds_table, wm_probmap, csf_probmap = utils2.get_maps_tables(
        sub_info, get_task=True, get_task_mean=True, get_rest=True, get_all_confounds_tables=False, get_probmaps=True, get_anat=False
    )

    # check if all-confounds files already saved
    subject_path = os.path.join(utils2.DATASET_PATH, 
                                'sub-'+sub_info['subject_id'], 'ses-1'
                               )
    func_path = os.path.join(subject_path, 'func')
    task_confound_flag = True
    rest_confound_flag = True
    for i in range(len(runs)):
        task_filename = f"all-confounds_{sub_info['subject_id']}_ses-1_task-TAU{sub_info['cohort']}_run-{i+1}.csv"
        try:
            utils2.load_file(func_path, task_filename)
            print(f"Found run-{i+1} all-confounds file")
        except:
            task_confound_flag = False
    rest_filename = f"all-confounds_{sub_info['subject_id']}_ses-1_task-rest.csv"
    try:
        utils2.load_file(func_path, rest_filename)
        print("Found rest all-confounds file")
    except:
        rest_confound_flag = False

    if not task_confound_flag:
        # get task complete confounds tables
        task_confounds_tables = utils2.get_confounds(
            wm_probmap, csf_probmap, mean_fmri, for_task=True, for_rest=False, 
            other_task_confounds_dfs=runs_confounds_tables, 
            other_rest_confounds_df=None, 
            task_fmri_runs=runs, rest_fmri=None, 
            threshold=utils2.WM_CSF_PROB_THRESH
        )

        # save task confound tables
        for i, task_confounds_table in enumerate(task_confounds_tables):
            # print(type(task_confounds_table))
            # print(task_confounds_table.head())
            task_filename = f"all-confounds_{sub_info['subject_id']}_ses-1_task-TAU{sub_info['cohort']}_run-{i+1}.csv"
            print(
                utils2.save_output_file(
                file=task_confounds_table,
                path=func_path,
                filename=task_filename
                )
            )

    if not rest_confound_flag:
        # get task & rest complete confounds tables
        rest_confounds_table = utils2.get_confounds(
            wm_probmap, csf_probmap, mean_fmri, for_task=False, for_rest=True, 
            other_task_confounds_dfs=[], 
            other_rest_confounds_df=rest_confounds_table, 
            task_fmri_runs=[], rest_fmri=rest, 
            threshold=utils2.WM_CSF_PROB_THRESH
        )

        # save rest confound table
        rest_filename = f"all-confounds_{sub_info['subject_id']}_ses-1_task-rest.csv"
        print(
            utils2.save_output_file(
            file=rest_confounds_table,
            path=func_path,
            filename=rest_filename
            )
        )
    print()

sub-20131
Found run-1 all-confounds file
Found run-2 all-confounds file
Found rest all-confounds file

sub-20209
Found run-1 all-confounds file
Found run-2 all-confounds file
Found rest all-confounds file

sub-20725
..\Pediatric_Anxiety_Disorder\sub-20725\ses-1\func\all-confounds_20725_ses-1_task-TAU1_run-1.csv
..\Pediatric_Anxiety_Disorder\sub-20725\ses-1\func\all-confounds_20725_ses-1_task-TAU1_run-2.csv
..\Pediatric_Anxiety_Disorder\sub-20725\ses-1\func\all-confounds_20725_ses-1_task-rest.csv

sub-21100
..\Pediatric_Anxiety_Disorder\sub-21100\ses-1\func\all-confounds_21100_ses-1_task-TAU1_run-1.csv
..\Pediatric_Anxiety_Disorder\sub-21100\ses-1\func\all-confounds_21100_ses-1_task-TAU1_run-2.csv
..\Pediatric_Anxiety_Disorder\sub-21100\ses-1\func\all-confounds_21100_ses-1_task-rest.csv

sub-21214
..\Pediatric_Anxiety_Disorder\sub-21214\ses-1\func\all-confounds_21214_ses-1_task-TAU1_run-1.csv
..\Pediatric_Anxiety_Disorder\sub-21214\ses-1\func\all-confounds_21214_ses-1_task-TAU1_run-2.cs

In [11]:
print(len(glob(os.path.join(utils2.DATASET_PATH, '**', 'all-confounds*.csv'), recursive=True)))

447


In [7]:
149*3

447